# Online Shoppers Intention Prediction

This notebook implements the complete end-to-end deep learning pipeline for predicting online purchase intention from session behavior data. It includes data loading, preprocessing, model training, evaluation, comparison, and visualization.


## 1. Problem Statement

Predict whether an online browsing session will convert into a purchase based on user behavior features such as page visits, time spent, bounce rates, and traffic source. The goal is to support e-commerce platforms in identifying likely buyers and improving recommendation strategies.


In [ ]:
from pathlib import Path
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, GRU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


## 2. Dataset Overview

The dataset contains 12,330 online shopping sessions described by 18 features. The target variable is `Revenue`, which indicates whether the session resulted in a purchase. Data includes behavioral signals such as page visits, duration, bounce rates, page values, and session metadata.


In [ ]:
ROOT_DIR = Path('..').resolve()
DATA_PATH = ROOT_DIR / 'Dataset' / 'online_shoppers_intention.csv'
MODEL_DIR = ROOT_DIR / 'Model'
IMAGE_DIR = ROOT_DIR / 'Images'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore')
df = pd.read_csv(DATA_PATH)
print(f'Dataset path: {DATA_PATH}')
print(f'Dataset shape: {df.shape}')
print('Columns:', df.columns.tolist())


## 3. Data Loading

Load the dataset and inspect the first rows, datatypes, and missing values. This step verifies that input data is ready for preprocessing.


In [ ]:
df.head(5)


In [ ]:
df.info()
print('Missing values by column:')
print(df.isnull().sum())


## 4. Data Cleaning

Remove duplicates and handle any missing records to ensure the training data is consistent and complete.


In [ ]:
duplicate_count = df.duplicated().sum()
print(f'Duplicate rows found: {duplicate_count}')
if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'New shape after duplicate removal: {df.shape}')
else:
    print('No duplicates found.')

missing_counts = df.isnull().sum()
print('Missing values per column:')
print(missing_counts[missing_counts > 0])

numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=['object']).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)
print('Data cleaning complete.')


## 5. Feature Engineering

Transform categorical fields and engineer features that are useful for model training. The dataset already contains rich session signals, so the primary flow is encoding and scaling.


In [ ]:
X = df.drop('Revenue', axis=1)
y = df['Revenue']
label_encoders = {}
for col in categorical_cols:
    if col in X.columns:
        encoder = LabelEncoder()
        X[col] = encoder.fit_transform(X[col].astype(str))
        label_encoders[col] = encoder
        print(f'Encoded {col} with {len(encoder.classes_)} classes')
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))
print('Encoded target Revenue:', list(target_encoder.classes_))
print(f'Feature shape: {X.shape}')
print(f'Target shape: {y.shape}')


## 6. Exploratory Data Analysis

Visualize feature distributions, class balance, and correlation to understand the data patterns and behavior signals.


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Revenue', data=df)
plt.title('Revenue Class Distribution')
plt.xlabel('Revenue')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(14, 10))
corr = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## 7. Data Preprocessing

Split the dataset into training and testing sets with stratification, then scale features using standard normalization. This is essential for stable neural network training.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Scaling complete. Sample feature means (train):', np.round(X_train_scaled.mean(axis=0)[:5], 4))


## 8. Model Development

Build four deep learning architectures: MLP, Deep Feedforward, LSTM, and GRU. Each model is chosen for a specific strength in classification or sequential pattern learning.


In [ ]:
input_shape = X_train_scaled.shape[1]
X_train_rnn = X_train_scaled.reshape((X_train_scaled.shape[0], input_shape, 1))
X_test_rnn = X_test_scaled.reshape((X_test_scaled.shape[0], input_shape, 1))
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model_mlp = Sequential([
    Dense(128, activation='relu', input_shape=(input_shape,)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid'),
])
model_mlp.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model_dnn = Sequential([
    Dense(256, activation='relu', input_shape=(input_shape,)),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid'),
])
model_dnn.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])
model_lstm = Sequential([
    LSTM(64, activation='relu', input_shape=(input_shape, 1), return_sequences=True),
    Dropout(0.3),
    LSTM(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_lstm.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model_gru = Sequential([
    GRU(64, activation='relu', input_shape=(input_shape, 1), return_sequences=True),
    Dropout(0.3),
    GRU(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_gru.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
print('Models built successfully.')


### Model Selection Rationale

- **MLP** is used as a baseline for tabular classification and fast convergence.
- **Deep Feedforward Neural Network** captures more complex non-linear interactions with deeper layers.
- **LSTM** is used to evaluate sequential representations of the standardized feature vector.
- **GRU** offers efficient recurrent learning with fewer parameters than LSTM.


## 9. Model Training

Train each architecture with early stopping to prevent overfitting and compare validation behavior.


In [ ]:
history_mlp = model_mlp.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1,
)
history_dnn = model_dnn.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1,
)
history_lstm = model_lstm.fit(
    X_train_rnn, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1,
)
history_gru = model_gru.fit(
    X_train_rnn, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1,
)
print('Training complete for all models.')


## 10. Model Evaluation

Evaluate each trained model on the test set and calculate standard classification metrics.


In [ ]:
models = [
    ('MLP', model_mlp, X_test_scaled),
    ('Deep Neural Network', model_dnn, X_test_scaled),
    ('LSTM', model_lstm, X_test_rnn),
    ('GRU', model_gru, X_test_rnn),
]
results = []
for name, model, X_eval in models:
    y_proba = model.predict(X_eval, verbose=0).flatten()
    y_pred = (y_proba > 0.5).astype(int)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })
results_df = pd.DataFrame(results)
results_df


## 11. Model Comparison

Compare model performance using a concise table and identify the best balanced architecture for this classification task.


In [ ]:
best_model = results_df.loc[results_df['F1-Score'].idxmax()]
best_model


## 12. Results Analysis

Review the strongest model and summarize why it is most appropriate for purchase intention prediction.


In [ ]:
results_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1-Score": "{:.4f}",
    "ROC-AUC": "{:.4f}",
})


## 13. Dashboard Generation

Generate visualizations used for analysis and present the project as an interactive model comparison dashboard.


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='Accuracy', data=results_df, palette='viridis')
plt.title('Model Accuracy Comparison')
plt.ylim(0.8, 1.0)
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'model_accuracy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, (name, model, X_eval) in enumerate(models):
    y_pred = (model.predict(X_eval, verbose=0).flatten() > 0.5).astype(int)
    ax = axes[idx // 2, idx % 2]
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['No Purchase', 'Purchase'], yticklabels=['No Purchase', 'Purchase'])
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'confusion_matrix_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 8))
for name, model, X_eval in models:
    y_proba = model.predict(X_eval, verbose=0).flatten()
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, y_proba):.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.7)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'roc_curve_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


## 14. Conclusion

The notebook compares multiple deep learning architectures and highlights the model that best balances precision, recall, and overall predictive performance. This provides a robust foundation for purchase intent analytics in e-commerce.


In [ ]:
best_model_name = best_model['Model']
best_model_score = best_model['F1-Score']
print(f'Best performing model: {best_model_name}')
print(f'F1-Score: {best_model_score:.4f}')
print('Notebook execution complete.')
